# LaMa inpainting setup

This notebook follows the upstream LaMa repository setup and uses the `big-lama` checkpoint published on Hugging Face.

Sources:
- https://github.com/advimman/lama
- https://github.com/advimman/lama/blob/main/configs/prediction/default.yaml
- https://huggingface.co/smartywu/big-lama


In [ ]:
# @title 1. Install LaMa, patch Colab compatibility, and download the model
import os
import shutil
import subprocess
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
WORKDIR = Path("/content/lama_workspace") if Path("/content").exists() else NOTEBOOK_DIR / ".lama_workspace"
LAMA_DIR = WORKDIR / "lama"
MODEL_DIR = WORKDIR / "big-lama"
MODEL_ZIP = WORKDIR / "big-lama.zip"
MODEL_URL = "https://huggingface.co/smartywu/big-lama/resolve/main/big-lama.zip"
INPUT_ROOT = WORKDIR / "inputs"
OUTPUT_ROOT = WORKDIR / "outputs"
SETUP_CELL_VERSION = "2026-07-07-capture-stderr-v2"

WORKDIR.mkdir(parents=True, exist_ok=True)
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, env=None):
    cmd = [str(part) for part in cmd]
    print("$", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=cwd, env=env, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    if completed.returncode != 0:
        raise subprocess.CalledProcessError(completed.returncode, cmd, output=completed.stdout, stderr=completed.stderr)
    return completed
run._prints_subprocess_output = True

try:
    import torch
    print(f"torch {torch.__version__}; cuda available: {torch.cuda.is_available()}")
except Exception:
    run([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision"])

PIP_PACKAGES = [
    "albumentations==1.3.1",
    "imgaug==0.4.0",
    "hydra-core==1.3.2",
    "omegaconf==2.3.0",
    "pytorch-lightning==1.9.5",
    "torchmetrics==0.11.4",
    "kornia==0.7.4",
    "torchvision",
    "webdataset",
    "opencv-python-headless",
    "scikit-image",
    "scikit-learn",
    "pandas",
    "matplotlib",
    "tqdm",
    "PyYAML",
    "easydict",
]
run([sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES])

if not (LAMA_DIR / ".git").exists():
    run(["git", "clone", "--depth", "1", "https://github.com/advimman/lama.git", LAMA_DIR])
else:
    run(["git", "-C", LAMA_DIR, "reset", "--hard", "HEAD"])
    run(["git", "-C", LAMA_DIR, "pull", "--ff-only"])
run(["git", "-C", LAMA_DIR, "reset", "--hard", "HEAD"])

def patch_file(path, replacements):
    path = Path(path)
    text = path.read_text()
    original = text
    for old, new in replacements:
        if old in text and new not in text:
            text = text.replace(old, new)
    if text != original:
        path.write_text(text)
        print(f"patched {path.relative_to(LAMA_DIR)}")

sitecustomize = LAMA_DIR / "sitecustomize.py"
sitecustomize.write_text(
    "import numpy as _np\n"
    "if not hasattr(_np, 'sctypes'):\n"
    "    _np.sctypes = {\n"
    "        'int': [_np.int8, _np.int16, _np.int32, _np.int64],\n"
    "        'uint': [_np.uint8, _np.uint16, _np.uint32, _np.uint64],\n"
    "        'float': [_np.float16, _np.float32, _np.float64],\n"
    "        'complex': [_np.complex64, _np.complex128],\n"
    "        'others': [bool, object, bytes, str],\n"
    "    }\n"
    "for _name, _value in {'bool': _np.bool_, 'int': int, 'float': float, 'complex': complex, 'object': object}.items():\n"
    "    if not hasattr(_np, _name):\n"
    "        setattr(_np, _name, _value)\n"
)

patch_file(
    LAMA_DIR / "bin" / "predict.py",
    [
        (
            'device = torch.device("cpu")',
            'device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n        LOGGER.info(f"Using device: {device}")',
        )
    ],
)

patch_file(
    LAMA_DIR / "saicinpainting" / "training" / "trainers" / "__init__.py",
    [
        (
            "import torch\n",
            "import torch\n\n\ndef _torch_load_compat(path, map_location):\n    try:\n        return torch.load(path, map_location=map_location, weights_only=False)\n    except TypeError:\n        return torch.load(path, map_location=map_location)\n",
        ),
        (
            "state = torch.load(path, map_location=map_location)",
            "state = _torch_load_compat(path, map_location)",
        ),
    ],
)

local_zip = NOTEBOOK_DIR / "big-lama.zip"
if local_zip.exists() and local_zip.resolve() != MODEL_ZIP.resolve():
    shutil.copy2(local_zip, MODEL_ZIP)
    print(f"copied existing model archive from {local_zip}")

if not MODEL_ZIP.exists():
    run(["wget", "-q", "--show-progress", "-O", MODEL_ZIP, MODEL_URL])

if not (MODEL_DIR / "config.yaml").exists():
    shutil.rmtree(MODEL_DIR, ignore_errors=True)
    run(["unzip", "-q", "-o", MODEL_ZIP, "-d", WORKDIR])

if not (MODEL_DIR / "config.yaml").exists():
    raise FileNotFoundError(f"Expected model config at {MODEL_DIR / 'config.yaml'}")

os.environ["PYTHONPATH"] = str(LAMA_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")

def run_lama_folder(input_dir, output_dir):
    input_dir = Path(input_dir).resolve()
    output_dir = Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["PYTHONPATH"] = str(LAMA_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    run(
        [
            sys.executable,
            LAMA_DIR / "bin" / "predict.py",
            f"model.path={MODEL_DIR}",
            f"indir={input_dir}",
            f"outdir={output_dir}",
            "out_ext=.png",
        ],
        cwd=LAMA_DIR,
        env=env,
    )
    return output_dir

def run_lama_on_pair(image_path, mask_path, job_name="custom"):
    from PIL import Image

    image_path = Path(image_path)
    mask_path = Path(mask_path)
    if not image_path.exists():
        raise FileNotFoundError(image_path)
    if not mask_path.exists():
        raise FileNotFoundError(mask_path)

    image = Image.open(image_path).convert("RGB")
    mask = Image.open(mask_path).convert("L")
    job_dir = INPUT_ROOT / job_name
    out_dir = OUTPUT_ROOT / job_name
    shutil.rmtree(job_dir, ignore_errors=True)
    shutil.rmtree(out_dir, ignore_errors=True)
    job_dir.mkdir(parents=True, exist_ok=True)

    image_target = job_dir / f"{job_name}.png"
    mask_target = job_dir / f"{job_name}_mask.png"
    image.save(image_target)
    mask.save(mask_target)

    run_lama_folder(job_dir, out_dir)
    return out_dir / f"{job_name}_mask.png"

print("LaMa setup ready.")
print("workspace:", WORKDIR)
print("repo:", LAMA_DIR)
print("model:", MODEL_DIR)


In [ ]:
# @title 2. Create a small demo image and mask
import shutil

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

demo_dir = INPUT_ROOT / "demo"
demo_out = OUTPUT_ROOT / "demo"
shutil.rmtree(demo_dir, ignore_errors=True)
shutil.rmtree(demo_out, ignore_errors=True)
demo_dir.mkdir(parents=True, exist_ok=True)

size = 512
image = Image.new("RGB", (size, size), "#f3efe4")
draw = ImageDraw.Draw(image)
for y in range(size):
    color = (220 - y // 8, 228 - y // 12, 210 + y // 18)
    draw.line([(0, y), (size, y)], fill=color)
for x in range(40, 480, 80):
    draw.ellipse((x, 110, x + 120, 230), fill=(86, 141, 124), outline=(34, 73, 66), width=3)
draw.rectangle((170, 210, 350, 300), fill=(188, 70, 64))
draw.text((190, 238), "REMOVE", fill=(255, 255, 255))

mask = Image.new("L", (size, size), 0)
mask_draw = ImageDraw.Draw(mask)
mask_draw.rectangle((160, 200, 360, 315), fill=255)

demo_image = demo_dir / "demo.png"
demo_mask = demo_dir / "demo_mask.png"
image.save(demo_image)
mask.save(demo_mask)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(image)
axes[0].set_title("demo image")
axes[0].axis("off")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title("mask")
axes[1].axis("off")
plt.tight_layout()


In [ ]:
EXPECTED_SETUP_VERSION = "2026-07-07-capture-stderr-v2"
if globals().get("SETUP_CELL_VERSION") != EXPECTED_SETUP_VERSION or not getattr(globals().get("run"), "_prints_subprocess_output", False):
    raise RuntimeError(
        "This Colab kernel is still using the old setup helpers. "
        "Restart the runtime or rerun cell 1 from the updated notebook, then rerun cells 2 and 3."
    )

# @title 3. Run LaMa on the demo
import matplotlib.pyplot as plt
from PIL import Image

demo_result = run_lama_on_pair(demo_image, demo_mask, job_name="demo")
print("result:", demo_result)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(Image.open(demo_image))
axes[0].set_title("input")
axes[0].axis("off")
axes[1].imshow(Image.open(demo_mask), cmap="gray")
axes[1].set_title("mask")
axes[1].axis("off")
axes[2].imshow(Image.open(demo_result))
axes[2].set_title("LaMa output")
axes[2].axis("off")
plt.tight_layout()


In [ ]:
# @title 4. Run your own image and mask
# Put paths to an RGB image and a mask here. White mask pixels are inpainted.
# Example: upload files in the Colab file browser, then set paths like "/content/photo.png".
EXPECTED_SETUP_VERSION = "2026-07-07-capture-stderr-v2"
if globals().get("SETUP_CELL_VERSION") != EXPECTED_SETUP_VERSION or not getattr(globals().get("run"), "_prints_subprocess_output", False):
    raise RuntimeError(
        "This Colab kernel is still using the old setup helpers. "
        "Restart the runtime or rerun cell 1 from the updated notebook before running custom inference."
    )

import matplotlib.pyplot as plt
from PIL import Image

CUSTOM_IMAGE = ""
CUSTOM_MASK = ""
JOB_NAME = "custom"

if CUSTOM_IMAGE and CUSTOM_MASK:
    custom_result = run_lama_on_pair(CUSTOM_IMAGE, CUSTOM_MASK, job_name=JOB_NAME)
    print("result:", custom_result)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(Image.open(CUSTOM_IMAGE).convert("RGB"))
    axes[0].set_title("input")
    axes[0].axis("off")
    axes[1].imshow(Image.open(CUSTOM_MASK).convert("L"), cmap="gray")
    axes[1].set_title("mask")
    axes[1].axis("off")
    axes[2].imshow(Image.open(custom_result))
    axes[2].set_title("LaMa output")
    axes[2].axis("off")
    plt.tight_layout()
else:
    print("Set CUSTOM_IMAGE and CUSTOM_MASK to file paths, then run this cell.")
